## `tmpfs` — RAM-backed mounts

A **`tmpfs` mount** is a chunk of host RAM that the container sees as a filesystem at a chosen path. Nothing lands on disk, and the data vanishes the moment the container stops.

```bash
docker run --tmpfs /tmp:size=64m,mode=1777 alpine sh
```

Three jobs it's made for:

- **High-throughput scratch.** Build steps, compilers, cache dumps — anything that wants a fast filesystem and doesn't need to survive.
- **Secrets you don't want on disk.** Mount `tmpfs` at `/run/secrets` and write tokens there; the host's disk never sees them. (Proper secret *management* is module 08 — this just keeps them off the platter.)
- **Writable space for a read-only container.** Pair **`--read-only`** (which makes the whole rootfs read-only — a hardening win) with `--tmpfs /tmp --tmpfs /run` so processes that insist on writing somewhere have a small RAM area to use.

`--tmpfs <path>` is the short form; the long form `--mount type=tmpfs,target=/scratch,tmpfs-size=64m` takes more options.

Two things to keep in mind: `tmpfs` **counts against the host's memory**, so a runaway writer can pressure the box (cap it with `size=`); and it's **Linux-container only** — there's no host disk involved, which is the whole point. Think of `tmpfs` as the deliberate opposite of a volume: a volume is for data that must *survive*, `tmpfs` is for data that must *never persist*.